# Notebook to demo User Defined Functions (UDF) in PySpark.

Here we are converting lon/lat decimal degress to web mercator x/y meters.

In [ ]:
# from spark_esri import spark_start, spark_stop

In [ ]:
import math

import numpy as np
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T

In [ ]:
# spark_stop()

# config = {
#     "spark.driver.memory": "64G",
#     "spark.executor.memory": "64G",
# }

# spark = spark_start(config)

In [ ]:
@F.udf(returnType=T.DoubleType())
def lonToX(lon: float) -> float:
    return 6378137.0 * math.radians(lon)


@F.udf(returnType=T.DoubleType())
def latToY(lat: float) -> float:
    return 6378137.0 * math.log(math.tan(0.7853981634 + 0.5 * math.radians(lat)))

In [ ]:
@F.pandas_udf(returnType=T.DoubleType())
def lonToX2(lon: pd.Series) -> pd.Series:
    return 6378137.0 * np.radians(lon)


@F.pandas_udf(returnType=T.DoubleType())
def latToY2(lat: pd.Series) -> pd.Series:
    return 6378137.0 * np.log(np.tan(0.7853981634 + 0.5 * np.radians(lat)))

In [ ]:
xmin, xmax = -180, 180
ymin, ymax = -80, 80
xdel = xmax - xmin
ydel = ymax - ymin

In [ ]:
%%time

(
    spark.range(10_000_000)
    .withColumn("lon", F.rand() * xdel + xmin)
    .withColumn("lat", F.rand() * ydel + ymin)
    .withColumn("x", lonToX2("lon"))
    .withColumn("y", latToY2("lat"))
    .write.format("noop")
    .mode("overwrite")
    .save()
)